
# OSR 508GB — Stage‑2 Direct Miner v3.1 · Canonical Runner

**只做一件事：`Runtime → Run all`。**

这个轻量 runner 内嵌经过 SHA‑256 固定的 Stage‑2 v3.1 engine。运行时会：

1. Mount Google Drive
2. 安装 `pyahocorasick`
3. 校验内嵌 engine SHA‑256
4. 自动把 canonical engine 写入  
   `MyDrive/OSR_WORK_SPACE/osr_stage2_direct_miner_v3_1.py`
5. 执行 engine：5-schema smoke gate → 1,788 shard resumable Direct Miner
6. 断线后重新 `Run all`，已验证 shard 自动跳过

Engine SHA‑256: `356fd6027f3c5c6525221e77794c2d376884edbaa724bdf9eddd88012fd35ace`

**不会制造第二份 508GB 正文副本。**


In [ ]:

# CANONICAL BOOTSTRAP — do not edit this cell unless updating engine version.

from google.colab import drive
drive.mount("/content/drive")

!pip -q install pyahocorasick

from pathlib import Path
import base64, zlib, hashlib

ENGINE_SHA256 = "356fd6027f3c5c6525221e77794c2d376884edbaa724bdf9eddd88012fd35ace"
ENGINE_PAYLOAD_B64 = """eNrtPWtz48aR3/ErJnDdLZGloMd67Q0TuUrWcm2dd1cbSmsnJbNQEAFKsEiCBkCt6LWqnDixYyfO2ld5+Jm7urIvl6RyWcfJxe8f44v28en+wnX3PDADgBRXTi5XV7eVWCTQ3TPT093T3dMzvI+tb7TY2YVzjzzMNjJ/J5xbYuejJOxk7FI0CBO2f8ZdtO5jq/4gHkQdv8fCwQ68YN04YY/E8U4vZKtxz992AeiJxB8OAWcQZ+F2HO+xfjwaZCk7n0T7ITvNokGa+b1eyoZjfzfuxImfRp29Ost2wwELD8LOKAtT+BalLO0k0TBzLaubxH029LPdXrTNov4wTjJ2Bb7yF52414O+RvEglS9Xsc0wqbMg7PqjXhZEnYwDR/A4i+OeAu3E/e1o4BM6B1m7Ms5244EbROmw548loPhqia9xWmdJWGdPpfEAeh/14XMf+lRnu36KHYXX/iCI+3W280w0hP926mwE7IuDMPAzv87S3VEW9SxJcAjQfsrgf8NAPRv7SRJfc4d+8vQozOjl05aVJeOGxeCfANMYSY8fXXm86a08us6W2WYyCq3woBMOM9akPzDQRhHqgt9LQ8tqXn5k7XLTe7zZ2lhbvwzPbRAMjwTD29hceaS55J1fazVXN71LANfyHj/jLdqWNUyiQVazv3j7+6bYCCnpxT4M2XYkXJMeN+w6M9tTACu78dyqGA+CyY46FvwDMWxevMiW2BfP/YRkIgWhSrMEZlgI7yLb8TMYzRPrrce81vr6JgwEpaVmz3diEItBNh+gMM5fGpNQzuMgCXjjyspqEzpKY12UuDmdeWZTE4ve+igbjrLU21+yrW9ebba+7QHqY5MxlrxvjsJkfMXv7KW21bp6eWMaLOciMdHb5zwWXYI/m1c3AE3vIiLyr8DJtQtrzfMuSqVtbaw+2ry04q1uPF7GSDu7Yd/3ujHoaeKBRnqLD54753bSfdtaWd1cA5bnAwP04iiBhA9Ktx96T+PIvCEMTTRroV0YgqazLaPbdZZ3qC0EuIuGgg3d8CBKs7Tm8Mf4L/GjNGQXol54Oc4ugEIHTdCFpNa1L0VpGg12QP2eHgGnAjXvfpJFXehVg10fHsI8WinNFvQeO+aiIKY1o0tuEvqBl4UHWS0cgGoC2WV7lHXnztmOYwkeBV2gMAw4LDColg8D2vDTFEwK4025OyEI8H6YRN0IZZ6BGSMVrIAaQKsePUD2x9cmw2/ZHWl6vQ6ZU7u9ZV9EW+ZnoySce2bXbrPlZbZ05swsmKu7IFpp+ES4vQl9WHIXOPbi2bNnZ0HfXN9cuShQQGaKKEKkpIDBBKZevHdv8CFONUdZkPC9cFBTM+IUqeWvtmygnI1Sjm6vP2Y7LjC4BnNVEGK3vxdESQ2sKxiFdBkZX2ckidBh+urkujoLsNX81hUwj83zHvEI8J5AZUWrpnVvMOp7YNWhg2466tdA0MQYKrFhVheWvLPnFr0zD95fr4IxLLDUBWES0RSCmQT6mgXOF/J010+CFI2syV0FuhnDas2wtwjUta+bYzklx3JKjqVRJ82TdvoM2ekOqE4WzqMCMW41GFkNhlbDss43L6xcvbhpGpzrZAnsgd8P7QZfifxBJwLmex2QX9/rwzLtpWEYePuLYCPrAj5O+n4veoZWdES8fGHVW13ZaF5Yv3heAgUh9y0EyNogyiIcZjhK/W3wZhLQDj/p7FL/XNYMIlhke5Gfhun8ThKPhunX5eqWACfAp4GlvbPrD3bCwJWNQE+BbJhCC1vKql3PNQpbvvvJ944+fAc4axNZfATfb3/wCT4SLSK+Bnj3k+fFpzuf/erO5z+lT5/8453Pvm+3D+uTGrr93R/P1pAAnEbqvV8cvfD2LKQUIHy68/nvpxH986c3bgPE8URzQPz08i/x09HHP7v90sfyEzyb0tDRex8c/dssDeWA+OnX04m+++tbr788C1EFCJ9uv/mLqURvvjYjUQUIn44hevc3r999/YNZBE4B4qc3PjuG6J3P35qNKAJOG/T33z/603sGqVs/eP3PH780f+vmH6rGzuGnietbH976pw9mp5jDw6fbH704jfadN391dPPdCtpFogJwGg9//0uTzu8/gf6VGMih7v7uObtNr8Bv8NAZ6oVgFf0EgHAlmsKNf/1ohmaO3v2lADx69wZ+mtQYBQ7TTMU7d37xitEgmoLv/KhsKghwGn9efOXuJz+egVQOCJ/+87nv5F/g062bN6ZK38c/ufv5G0Yj8L1K7BTgrRdfuvv51J7ffv6t2YjmgLdf/C0SrXPk1/8gn71xc6oN/fCVo9f+7ej9901NfPvm7VdfKJlRBTuF4K3XXzh69XNTXm7euPXzG/O3fvbR0VufFYkK+GkU337p9gsv3wNFDj910C/d+eHPZ6eYw8OfWzdfm26BXz565V9Mbt54F0zb0auvlI2whIVPt9+aulzc+eNv7jz/3mx0BexUDnx29O73ZqOWw979+Od33vjp1NH/+0e3fvDqjKNXsPzTcXr27i+PPv7tjKQVLP90HOlX3gRfaUbSCpY+PT/db3jxzncLmvX++7f++Kfbn9wouw4Sln8SYii+vP3DqbP50Tt3Py3YuXc/PfrBf8zfevm5W2//rjSjHH66v3ZPFAX8NFv86Y9vffSzKorVxkbAS4pt4SCnfn/Yg+UEYK6rpuwhxIIQD/Y9/EDxCbxfyvti7/TibYhHJRy8PbOwoL3vj3pZVCZxVgNJB9FwGGZe4gfRKKXFDHt65uwCwRxah5YlUhOlXEghVVF+fy2BwJznFFSLlH8IRv1hWisHOxBGDlII4z0/7UTRMq2pdQgaA4gglpecvN/FFAW9cOi/WgS4SsFWIHOf5YALI7lStx0rhGYbRWJXKdNCQ8YPs9Ky8qSQmX0ps2tqBsaCQbA8B4GEavH2U4L3SZiNkoHOXHhXzc00TjJvLxzLwD0NIZD3szhJl2t2HQW2Qe3l/UbBWTr7AHRfJHRd/oBPaqFPOZrj0jDCmhyE5bi74UEQ7YRphjmI1tXLMveX5wHnWanhrcbiA21r49GV1nmv1dxAkTm/1uJYeRaP4nfburCydvFqq1kB0fWjHnADYIqkZkloaHRnyn/U9Kb1IQ38YbobZzxD6JR0RJvCHGsmvSjphKPSId80pDSnu8WTCm2nDMiAScB7E15MiAJvjQYsieMMoeR4tZTH/ZTy2AcDGGDyJdcVdpptj6JegPsEnd0w4dKNCQuPpxtqqaPSoimmAi/Hg1DLiHJxt209dRqltKMy6IS1tI5ZHy2FmoIswBMga6mvoG/Dng/Q9pPJkwMU/CdxQrSnFc8OFhbwqe3oaqdtZrgy6wLAly+sAmzquB2w/924F6DQy1wI9EBLF1MaVKVJ6mwLpkQMSz4Vik65YOA77rPwNLA+aaCggKRwXExADf0MloCBl8We0lXMKh1auerCO1rA+HNKWwMNTFybzesEkJ/wVsuMgiC5mG0b1hx9YtT7Ykr7cb83koNo9ofZWKNP7Y5pAKaVkV3dUo/a0BkA50zk63A+RXqsBnDbcdyrKeByIFfngZwjkWnpBrweaLeOx5d0mijRQ31XEPjXR6n1R1k85wcgGiwe9MbIkAgFldjiw7NwDluFtSRMmJinuqA2GvTCFPf/QsELMDHgJHSiDAjFwyzFZGrMeOeJDNcm+O5KlgWkeF5hGOKrmiNMeOq6pxjrOOyhZbbEQBp0TuXTWGrCxS3PQaCR4MzpEgkAQZHCPUgXNxhxFaqVaGhqi/Ko94wgTBnT905yROpcDGwbjEIdDIc65BlzBu3yeagcWr76d+0v3vzn//rwBkv30FcKDJZTj8BrUwM+ZHMPsev0+NB2ju0QbQ5Vaih2r+rF1rDNvrJcpVSVimW8pamwV2gicLMYTGY8YH43o21qLV3cEEP4SkLjsSuoXJ/QOUTBvucsgScmgZwtkwaojU8ZMDLa4LyEQa0KzSVpcuoM/i7DPOOm9H6YpKFckYHbcv+U8wy1E1QGOLCsbxy7K/K5EDG1iSc7krNckXBBy71rcQJ9qzP8/4SRScOi8Pr+Hki+1mLufHZBMrdx1U3CnfAAOpmEbifuD6MeLC7P2u5TcTSowbMw7fjDEOW61FPpOCb+NY+vtWkNfQ2hY2lBv+iVNAsmsyQjQLu9KDjAQQp7lU+Vg03nQ8NduZquztRk5oP/CRTIaBMtNsf1ktNzwDlYNFDGUQiugmpF8baeE6srWhI5Z6PseR97Z/LU7YIPVdlN0QR0su/SklJbcCo6VT3L/GFb62jfpa6iePZdNJBiXmBlGA1DbwAecRioGRJ/RZfuYxeg97gObKxcapaWyDqLQcx7YHgpLBGmnvYoYYHphK4g0jzAIDNsMJWXYOCHjnrcCHL47TjbFe9JhxUoJ7I9xlECT7RikhouKbmewHsatpgbmhfkuxhSzmROCxfxtlw0agU8ubKCz47RG7kmRjt9Wk44JZS2vrFt3tcMRj8VdsHvbwc+O2iw2lztYGupDbJ3sLXYhnnBP/jfJamj+G8Pq0WWYZU3JKlqfKkpP2JOsAv+YFxDFtcQ9BvLbC/FBZXQcXndC7neetBD6iWSw3ZNeRNrnKTbKFlkRJnCydx1RmYS33P2IG6RQcQW3cvlmFZ561S5iGLzVH7P904vcTc/N6ACUjNTJiQCGNUvtm6RSLmZzc2i1Gh9r/UsBR58cxYMEm5anmZ+4A+htZRrHkRvaeh1UYtTpf16GM2j9BRrJ/INaw5vE+u0XXa5YY8vKQy222DYhzW9EadEx8Pgq0RMNkEEqFdyTsDP2DqQIRvJzAHKykHaxlAPvMOwg1ZE0IJQbUg+H88o1Ww07nZdBIvyK5a/xElolx7DFPYqHqfxKOlUPA9i3PwlxPgAndQxfn56BPYoG3uqDYgt4u00TPaxn2YHsxFYp9pByrXhgFR7EsfcLCZP1kGCo8HeIL6GZqlIeo5VMwVdAYE1Marq2lcHEltKEmGjsLDrAp3X1gjao37fh0BsmfFpU53nC8j2uLYl2VdXVUc8oMZdJT/1MKA/4NE91zzX39nJPTgqCFmu2VjrhSTIZttaVgzLD+C9qqrAZkZ9HWJ7nBGJFAJUj75UAGmsBlCd8QDbjZJUNSp6SSmlffQ5U2OM1GE+tg5YIcxNbPFsCY0RzAsug7yasGZy0UFdXqeqgiHWIAwyn/JtWFAJ4dIIKzYIAdyP3Tim3BwukSn4sNBoxus5SJTSfgxGFYaauRZQE77HPc2Os6VzrO0CiX6EzhrRBl+CDH1x2ntxZ0s02J7OqGJzHBrHnXGhqAVJPBSOrGNUuaz0emDupOQbkpqyXR9YJsNGZQAxqFZcVwPYKs4c2RnsnCErunAZOtnW7O8DZH/RNsxR3AsO6gDrIE+zlEwi6DasY6hZwCPMPVIu68CyrjRb3mazdcnDD5SmE0VDes5KZcvbPBqvyJPX2RKsKY9cXH945aInic5AqphRr2NGHShdunpxc+3ee1aZgK+zs0Bx4/LalSvNTa+1cn7t6sYMtCoz9XXM1KOftLnSegSoPbyyufqod2ntYSC4uGRdWrssHon6q3PWpZVvmY+WFu4/x5dELn48Fzk9/4bSfuLkm1hkC7nktJAtBr8F7XC6DNYJPmD1FhZv2U4hiSw6ngSY066BZIqGh/4YV3ClliJoBYCtU1zOT7UPn+XfpbTnTwyFxMe2TgSnioMp1TjVdg71XYjqQYpOFRPjxpC2Gkv3t/m4ttEZIm3zwJBpgxvgM4xN/IPaYp3J/mj1dDLO3N/xMGSUsO5CnXWhCwJeN2oOm+d0OSb4jiB2WDddlKyvssWFpfvFH32w2IIpceCig5k0RY53V5CfVz10ZKhKAbEQ9hrGrSiPuE2BEaonfFrxDV1buYJfI5lEeZRIUm65u1jTko/yfVeXW60tEF+OhGIrnzva9PqCoQt8hOjLAphjdtMBB8RUc468jcgG1jRS5L6D5ayiJD1z/9qW39gWUoPLRrIfR4nnB0Fte9TZC2FMGB4hzXDA61fB3a+zZLCT6zm2zKEd9g22p4Vq9FDGFkjIqQixn8IExWDHxVp/nOFFvTUjV/cURUFmBMPb2HoKWLaIDjG2wocDIXqUghZQ9MEnEaS0Ts6OxxeWAJ0AMIf518Go16NJI3Woswy4m2tTXVLxdqMM3sadDtgYUMpQPKAo2is+5lnVOH8MdNG0k6UOAY2be/VVLnLUpmWGGNo2Ly+f9DBXxaswzcMA+mYtX58bjCsv/9bWANSyLUDUdx3IdDYksYI7WmzU00xFIzc4hgHRkJTfTdaoUWmhNHB9LnNo9UgHNSZawBrPdGBTDAS0+dDoNUbWGCyEnXgQIDy3lRgVuPifGqp0tqDj5FIlyOcPjG7HvVF/oHd8i8dObZMNJGnwllIr8rtOqCCUErTwWMeoFGaJV/nS7Hj+EssT7GeflanHPQiE9snFBlOyj/GaDiwTMlrxBHiHMepGFpb8QPL/iBZ68ZxGpu0YuhoRYAuHBw/XT0LyIXssi/lZqajfH2VUu8wll3sHOb5tqCUOaa84DAOiYhy2rvaSk/ozp7oII8cwLYUoteC+DAiIB4sWdxZrZOo4nIeuNHVpGVc15edku/JYD61WqGUUIUofQO6OwLN7OFyC4CL/lks0bqCVfRLLMKcAIw6difR5QbhK7ytFsASli1bppTFhlflJ7tNokzAxi8mBYOUqtWJMWp4VFI85xoJihtJ1ekpD0M2UemqaI40ERLTL4uSc26I/uQuDhqaqGENzUku79sq3nez92qWSDbVlY7qoiw9AVL/4gHSJuCTS4aCn3Sv8iB6KlJAiYusCvNUMqdo0TaKMsojxEF6o7SfkElu/fPHbFEXKHPZGlow6eM4nYP0w83HDHZ28JNwdBwmV+vR83Mui04voEqKpECFm6qrkMckwbZN0aZPC2xaJ99zlISnH1W256DZwYSRrvly24aM09LJdrORJRZkGZ1GucIihFMnllGoLjpvF3nAsslpGthnHAX0lvEYxJ1w0DXJzmfxeTOEbkvjQcgmhnEgWHkqF31WC/Iv4YRVUT+CXFf+d2E8rEtJmQ/aOL1bLxtALu7AiwtBCkknRiIZTDkgqYhFnwlaqEAitpfLUFmzN6WW2WBpdLi302uCpYcDgtQhWRHhkwNKGj9f3h2Roq3a59D1JRaRy00PRKo9I7ambHZV7E/oWkCQiNooL/RWbSmJLA30ZRaPcai5aPAoXlLWSFGuSVOtlKyUGVyyXBXjkuNa6U0I3hFlDVuEb9625DBeZYGxwefA/ZITWXKNS1Sq1EndB2xUSJnwWY/9sWW9ja6FdxkAfTO0UFP8ZJbn5NnE1rKzTLRcUUZpN22W29Zy4QYNLbeCJ7SughiOqhj0uZjOBRYXuxODNhL6nQM5AVRKQR1tCIqYhYNemeR0FD2SmXBtvdornYVg73QuZ1FNhSnK+6HlNZWgmYKOJ4ykLQDTM2yRZEH4Iln6fb15otlp0ItW70KSE18oTmPSyJ80gT3NhQDAh65Vv4ZZJHJb1RDmvx5oZM0tU2T190dToVY+F8ktTyJh9qoYsZ/8neAiDnfpx6zQsHUuYZ8I5zE25g4+WFhqVVs8H34RHsflFGBou7ik0jvU1tmpIxplk+rRgoXJaaHtNL38ie1iE8grmM1+BCL9dhcNbntGQprqFTP/ftP1fMG1f1ticwIJUpDkmi+W096gw1e+rduZOZjWCsMeDLMs6Pgz6y4U+Jw93vlSIo+3WPsirZWibHPfZGliYXNx45/vpuP0Owa3cfLesjUvrjzW1UxzqyINH9GwNYKZjExL8iQ0+nXS/CJB+4H61144TIdIv/JE82MGfUfF6VBdZQpZvcFOoj+OX6S+xi/7kwGanmb1ss6+yry3oh4i6NnWHbV2PTi8ezl+nmyEUPeewzQqGgD3LyjkVkf+QNxUZFW0zZfsqeZJbEYMv0s3nX7VCWxqQIf5YPlBwTGyegrLrskzMzMwXTK79LJd5QkCjyHFOGYpwqo33YJQQeQpGb0lLpJfb4SlBPE0x6te0vvFcdtsVNRWOZrjEHlXhwie8OErfyDYESDLPXB1nWudm2no54dJm8ztgGqSStVAb42Fxgu0v3vkR8ZQA8RiSpqB4SsqQleOPIBnwJzqFVLec6k4ops/aC4lw4m4IcxHvqaIAXaFpiHQ04CxPS4Nnlb+XjTv8Ih4JgLfpHBS1hD3EFvLyPJO+Vr1TtDviMTc58v4qLvSi3wpG4chDQhJiYjlbvh+xoew8w0GFgcsuwFLFFusPnjs3x808ZV+jlG334s4eQujlD2uDFDf32HVjVk9VzOopVRKh1ppztNZ0Kxu8FmW7DL+GmF4CjejsDWMYb6r2tYGFGHTvFsohxFJdOiE4j1bJLA45pI65O8/YYneZ93cKVf1A4USCgpqfxf2o43FRxsvu1DnNOgvSTBV0ryZxms5Redw4RY889buhfMm3s4gGnr3g1wUC0BAW/OEYH8EKEerYvFnoMBocnuYmIl5GlR/mlW+2w4cB/XER/tC9HqeY+xhGQQ2GQ5iAyKeO7ocTdBAD54gqvmoSH2XYJTBEkjst+mpH84rccGMwrzXVtTqzr2HxaOm0KxrprhlqKWvAmdmdcLZ1wllW5eJBn7n/MYszohYJup7QRe4j0/UBKPZojRSGq0CgN8mswx0m8XaonxeudbUW8NwQAojCMH7Hl4OngvCOr6oDQeaBQV2klJYxfldbh+JeYRpsrdE4PzCpDQmlGoUWRVKIai6YfMMFQuleb2yexaBDKtWMNLtvCJI+/oqtTIOB7mjQiwZ7NTPaqr7/0cxNpsLe0LlVTx74lhGAVmFGJ3LKNunY6/yEXaGbJo9TlnuSGlCNoswUGq0V4QsiJK6JM6BwnVOQhWIVwjDrVSbjlnYkOXrp8WQKsvoc0QxnbDKK8sNypNw1m9KS4ZBpDZqOWokALs+F7npGAe3cokPUqktoptMz3IwKUlpxzTFu8ARptDbWr7ZWm95663wTj89fL1yt2GAQxFZcmthgi4fWtTjZ4zdE5iXQaDNBBcWrLYgKOUtiWOTpVkOmXsmppHMPej8cjbL4VCikNqnq5cttZ1IVtdURpSiBtx2C6qK1xeCCGDNZ9dUesqciTNmnPLy08gtce7gVPOYE8VBLsVXo7Lw4DCPo4HpFISyeapvYhKraO85KCbCaOBvnsL9jZxfIia0+6wqxbnCA0S5eKdkmA9cPg3lx9lVLXpk7b4X4sgvebYHSDKGy4WY+y48z5MW2+QWLdLWiXnowa3BtrqHVMS4uo1UadeyyWnFGthVfm+Mn6/pRSknaBlNhst4sDh7arRxs6fBsYaTuaIjHp0tRK7fpDTLohYBSM7kiZi3fCTEZRYWupWelmBhzZyDrfuaNBhFmZLUKECOMzU/UVrvQfKj1ygVXr5fARbnSp9cnvjvJe+hqbsNxqROmciVa6sOc08rMx4lTJnmBFcfVUiGn5LtTWiqkkkgadoptF6oeoXV3saujCl4ck0oRXC9tLmiC2Gy11lvFPs1c/npiMfyb529OUkLLb4DCOj7RBK/pmzUzxMdETvyx2lecQlNxKm9xKiRoBOLM1zdVZ2gqrgeoTmrhq52OK667r2npha9ReiEJg1EnTBr8VDNei96J57TywdOMn/PJtzAQRo9/LEucBKqsahTvppUsChCewZ/0dmJNo3g/rVTRACnXK/L8vxgm5cWLZYrxHr9hmr5t+4GZWa+qPjRPKOmXM1EtoDi/IRk36ymE/9UHEI7z9L5cJKhx/X8k9Qy+aMrveb60trGxdvkRkSyzKzPKpoP31wxSixe5zOLTVt8i1OK+n5HKkK6XkcyQwm9swhd0XjpWevzFNxzq7PqhVqpWbQ9K6MUS/GoquckoEaiu5BdkjPyKrGRXqEYZP2GUL0wg96tOx5P23HTYiwAPq/1Bu83AtMJ86WUPqHz7hQ5pdVXSqundM8rjp3SPAiNMeWJinYOXUzhl03hsNUyl0apMLJmm+UuUxUztZTVK4Tzr1N1snVOKzRVnEegKK5OH02pUZmOUvvrUrdl4Mm2jHw/GLS4sLNSLLGBfZUW/YjZv9W9meqv28CyaEVH+D4vvTNevFa9eyw8v0KmV0qCOq5JUlZFVV6lVH1JCRS/YTE2Gq08tFfAmV7vq6EJJ9Lo8Uw+LtZt4LZs4ViPZK3+75Lyf+RcS3MFQzHKMfBKhbenG3ui+cZ0BT5tRNqmiAf5jB8ZRP7yxx8NiQb9eerwNj7crjl7RGpgfMNuXI1O5KG57hcmvMM7Sllptc6TlFkpjc3BNrnJXqSTdGK1lnoxQY61rA6xXjIpuL+FL2wwsVPcrBnylRFbi33q1oFfeFIbCwpEKsl0W0zKzBaLOa23JnsRpnfJUJmsRw7EsLvGirg++KLfIZqqTWl2/enlzQ/wYkl6zQwrBYyb+80er65OBdWkwkFYurq1sTMbjA9QRNlYuXbnY3PD+YWP98sUCtFmFpNQd3ceeLa5lqMIrr3QS5+GV84RRaohiRk9fGMRvR9HvI1XhVP4SmPnDU9L24PEe/MGkwgTw6Fjd+UJWRIIazC8ASnWRsEWeF8AtctjJVzeYjTuwx7jtpXMR4jiFaX+dRqXjMdFZKnh1PMdQ0zIKPFAspxPwJDzdyaqPSRODGUckO6g7KtoQTtQfKVjTKlk06Tp5GYv8Ua38yhlR/aHKUvLYXvw0lF7QovVBVLPw3wfLRvllTDb9JNWEZFz+q14Nfp+q/C46fQ/p5XvJ52m/wMUve/bkKXPcf4JRTgKL94Qxl2xxJkEia+LE6/OfVRNYRZ5JbJHVo5Sv1pOKn6USCFhp24mT4Sj11CFl8TMlAqKrakYDYOk+t3n8J6MC+RsjAjTmv7tn3hivGXAsDs6Sor0pnws3oItmZNJBcBPLMFSGq1hpvAWWYYiOPTCNGJqam62ULLdAkBppuJJUvSOyNUGUyN4Uaod0lIrbuwXWbHd8Cw9C/iwX/7G7EF1trSzrKqz1nKvzMkDOYvFzYeIyFrzeiXL0c1ikoB2OpwsFbe2HA5K4H+Mh+7DH7y8L96OAvDa6MhhP2Oe/6xUNw140CF1bz8IoExPKIgUi3AqT0UD9oOnXeaJG/HQav+wY9wnJ0ggdmuc5aAmThFkyFk3x6+C01XVqCSDZp5PbzAk1d9pQixf+8//xH4v84rnXlhhf5hkt81p9HttsXW3axm8PrMjqNm3lFAy4FiYhkyco+XUHkT6VeDdX+ecH+P3DeVcwSTYOM7Wb7JrtP65NC246S7uHm81kKU3wh/0ABUjMmLyb0TB5OngrnEMpoI5LUWjkG9uiUlxmk69FwIztUF6e7NoVRYxqcjbjIYWwQhMa2kVl0o/aDf2gdobuvDLQjAy/gcmdqio8vL4+oF8YjZNxQ/Ws4ib7xQW+seBfy3VJnjrH5MJpLD40dxDuY+fDLhZKsN2wNwyTlK4A/zoL4lDc/80vykXp6I2BGv9NwMGY9VDb5bUVsvQx7u3LjWy+8NQ5hPC8+sYlkkpjjd+CFCkMXPP5Z4f9fSVknsUgWGqG/16JnqrvU8XZYvEqicfCsbxBcYMkAcc6GkQg4zRMGkkAyxbvwyHYTl5SeWheqt93I7xAb0HcatQNqTwcllBYcE0W1PND2XXtxkdyHfSrMjCLvzyVl5a6/lBdli7SOtpFd4X7scwzRpbkkHj/DbaAuiW+PXRcaQFn4RqS4kzkiFNuWKi+8yPudlPq3YJlXmcw7QaPE16NgB/LVyNwLt7LxQjANNHt0+JyBMki3K7hjChsJQjw5QK8Nf2EOC87jJgkyuYEpXIJiZkTRAFomBc3RM6WINd2wZuiaqdSwrpOmDydN+pjPVVYk5fCVhzm1G4PEBUeIkd+gMW7uNTnazW/CxOFmS1XnwYzdlcK2lG6LUIjB7pdgD5JBQyaxbz2RRJcvl4gfShaXr6e92Ba3UvhDi0zYSt02qo8f8gthjXxyCAXiS+/518+q5ePrNw8JYapwqUoEFX1uy0wZDhtoE14Dt02bwQ1LaXDqAKNfkiDDClp9W6UefJeFO3HGvgM20K+vk1XnOMl+mPwdmO8dQXw1JWdlAwrXUdOV5Uon2aOfpUE3AVLbhtC3IJDRQ9I3tlKP6vOeGTkGj0Q26/CZ5++CTtpI1YhO7NtM+bwx/ycxF+pUpeufFGXjOi7N+bFVXyXrJz3blRcU18+7jvrnQmaVkm+TDvqa6qZd+xh33s6rVtSQ71L05RRXK1FTiWnDkycKP7CumB51SKXQLoTbIraONZ/A+c7oGE="""

engine_bytes = zlib.decompress(base64.b64decode(ENGINE_PAYLOAD_B64))
actual_sha = hashlib.sha256(engine_bytes).hexdigest()
assert actual_sha == ENGINE_SHA256, (actual_sha, ENGINE_SHA256)

drive_engine = Path(
    "/content/drive/MyDrive/OSR_WORK_SPACE/"
    "osr_stage2_direct_miner_v3_1.py"
)
drive_engine.parent.mkdir(parents=True, exist_ok=True)
drive_engine.write_bytes(engine_bytes)

# Read-back verification from Drive before execution.
drive_sha = hashlib.sha256(drive_engine.read_bytes()).hexdigest()
assert drive_sha == ENGINE_SHA256, (drive_sha, ENGINE_SHA256)

print("✅ Canonical Stage-2 engine archived + verified in Drive")
print("Engine:", drive_engine)
print("SHA256:", ENGINE_SHA256)

code = engine_bytes.decode("utf-8")
exec(compile(code, str(drive_engine), "exec"), globals())
